In [4]:
# Kill everything
!pkill -f "mlflow" 2>/dev/null
!pkill -f "uvicorn" 2>/dev/null
!pkill -f "cloudflared" 2>/dev/null

import time
time.sleep(5)

# Downgrade to MLflow 2.19 — stable, no security middleware issues either
!pip install mlflow==2.19.0 -q
print("MLflow 2.19 installed!")

^C
^C
^C
MLflow 2.19 installed!


In [5]:
# Complete clean reinstall
!pip uninstall mlflow mlflow-skinny -y
!pip install mlflow==2.19.0 --force-reinstall --no-cache-dir -q

# Delete ALL mlflow artifacts
import os, shutil, glob
for f in glob.glob("/tmp/*mlflow*"):
    if os.path.isfile(f): os.remove(f)
for f in glob.glob("/content/*mlflow*"):
    if os.path.isfile(f): os.remove(f)
if os.path.exists("/content/mlruns"):
    shutil.rmtree("/content/mlruns")

print("Clean reinstall done!")
print("Now RESTART THE RUNTIME: Runtime → Restart runtime")

Found existing installation: mlflow 2.19.0
Uninstalling mlflow-2.19.0:
  Successfully uninstalled mlflow-2.19.0
Found existing installation: mlflow-skinny 2.19.0
Uninstalling mlflow-skinny-2.19.0:
  Successfully uninstalled mlflow-skinny-2.19.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 274.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 81.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 308.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 254.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 231.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 254.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 150.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 148.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 229.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import mlflow
print(f"MLflow version: {mlflow.__version__}")
mlflow.set_tracking_uri("sqlite:////tmp/mlflow_fresh.db")
mlflow.set_experiment("product-review-summarizer")

# Run 1: Overfit
with mlflow.start_run(run_name="run1-3ep-lr2e4-r16"):
    mlflow.log_params({
        "epochs": 3, "learning_rate": 2e-4, "lora_rank": 16,
        "lora_alpha": 16, "batch_size_effective": 8,
        "quantization": "4bit-NF4", "optimizer": "adamw_8bit",
        "base_model": "mistral-7b-instruct-v0.3",
    })
    for step, loss in {100: 0.843, 200: 1.004, 300: 0.710, 400: 0.670, 500: 0.357, 600: 0.158, 700: 0.300, 725: 0.437}.items():
        mlflow.log_metric("train_loss", loss, step=step)
    for step, loss in {100: 1.547, 200: 1.475, 300: 1.596, 400: 1.570, 500: 1.783, 600: 2.119, 700: 1.915, 725: 1.903}.items():
        mlflow.log_metric("val_loss", loss, step=step)
    mlflow.log_metrics({"best_val_loss": 1.475, "best_val_step": 200})
    mlflow.set_tag("status", "overfit")
print("Run 1 logged!")

# Run 2: Underfit
with mlflow.start_run(run_name="run2-1ep-lr1e4-r16"):
    mlflow.log_params({
        "epochs": 1, "learning_rate": 1e-4, "lora_rank": 16,
        "lora_alpha": 16, "batch_size_effective": 8,
        "quantization": "4bit-NF4", "optimizer": "adamw_8bit",
        "base_model": "mistral-7b-instruct-v0.3",
    })
    for step, loss in {50: 1.631, 100: 1.544, 150: 1.601, 200: 1.590, 225: 1.590}.items():
        mlflow.log_metric("train_loss", loss, step=step)
    for step, loss in {50: 1.592, 100: 1.592, 150: 1.592, 200: 1.592, 225: 1.592}.items():
        mlflow.log_metric("val_loss", loss, step=step)
    mlflow.log_metrics({"best_val_loss": 1.592, "best_val_step": 50})
    mlflow.set_tag("status", "underfit")
print("Run 2 logged!")

# Run 3: 2 epochs, lr=2e-4 (adapters lost in crash)
with mlflow.start_run(run_name="run3-2ep-lr2e4-r16"):
    mlflow.log_params({
        "epochs": 2, "learning_rate": 2e-4, "lora_rank": 16,
        "lora_alpha": 16, "batch_size_effective": 8,
        "quantization": "4bit-NF4", "optimizer": "adamw_8bit",
        "base_model": "mistral-7b-instruct-v0.3",
    })
    for step, loss in {50: 1.436, 100: 1.433, 150: 1.430, 200: 1.422, 250: 1.467, 300: 1.465, 350: 1.470}.items():
        mlflow.log_metric("train_loss", loss, step=step)
    for step, loss in {50: 1.436, 100: 1.433, 150: 1.430, 200: 1.422, 250: 1.467, 300: 1.465, 350: 1.470}.items():
        mlflow.log_metric("val_loss", loss, step=step)
    mlflow.log_metrics({"best_val_loss": 1.422, "best_val_step": 200})
    mlflow.set_tag("status", "good - adapters lost in crash")
print("Run 3 logged!")

# Run 4: Previous Winner (adapters lost in crash)
with mlflow.start_run(run_name="run4-2ep-lr1.5e4-r16-prev"):
    mlflow.log_params({
        "epochs": 2, "learning_rate": 1.5e-4, "lora_rank": 16,
        "lora_alpha": 16, "batch_size_effective": 8,
        "quantization": "4bit-NF4", "optimizer": "adamw_8bit",
        "base_model": "mistral-7b-instruct-v0.3",
    })
    for step, loss in {50: 1.440, 100: 1.427, 150: 1.425, 200: 1.419, 250: 1.463, 300: 1.461, 350: 1.461}.items():
        mlflow.log_metric("train_loss", loss, step=step)
    for step, loss in {50: 1.440, 100: 1.427, 150: 1.425, 200: 1.419, 250: 1.463, 300: 1.461, 350: 1.461}.items():
        mlflow.log_metric("val_loss", loss, step=step)
    mlflow.log_metrics({"best_val_loss": 1.419, "best_val_step": 200})
    mlflow.set_tag("status", "best val loss - adapters lost in crash")
print("Run 4 logged!")

# Run 4 retrain: Winner with adapters saved
with mlflow.start_run(run_name="run4-2ep-lr1.5e4-r16"):
    mlflow.log_params({
        "epochs": 2, "learning_rate": 1.5e-4, "lora_rank": 16,
        "lora_alpha": 16, "batch_size_effective": 8,
        "quantization": "4bit-NF4", "optimizer": "adamw_8bit",
        "base_model": "mistral-7b-instruct-v0.3",
        "lr_scheduler": "cosine", "warmup_steps": 30,
        "early_stopping_patience": 3,
    })
    for step, loss in {50: 1.520, 100: 1.273, 150: 1.290, 200: 1.284, 250: 0.959, 300: 0.974, 350: 1.004}.items():
        mlflow.log_metric("train_loss", loss, step=step)
    for step, loss in {50: 1.452, 100: 1.427, 150: 1.425, 200: 1.422, 250: 1.461, 300: 1.461, 350: 1.461}.items():
        mlflow.log_metric("val_loss", loss, step=step)
    mlflow.log_metrics({"best_val_loss": 1.422, "best_val_step": 200})
    mlflow.set_tag("status", "winner - adapters saved")
    mlflow.set_tag("adapters_saved", "yes")
    mlflow.set_tag("adapters_path", "adapters_run4-2ep-lr1.5e4-r16")
print("Run 4 retrain logged!")

# Run 5: rank=32
with mlflow.start_run(run_name="run5-2ep-lr2e4-r32"):
    mlflow.log_params({
        "epochs": 2, "learning_rate": 2e-4, "lora_rank": 32,
        "lora_alpha": 32, "batch_size_effective": 8,
        "quantization": "4bit-NF4", "optimizer": "adamw_8bit",
        "base_model": "mistral-7b-instruct-v0.3",
        "lr_scheduler": "cosine", "warmup_steps": 30,
        "early_stopping_patience": 3,
    })
    for step, loss in {50: 1.450, 100: 1.213, 150: 1.227, 200: 1.252, 250: 0.823, 300: 0.824, 350: 0.844}.items():
        mlflow.log_metric("train_loss", loss, step=step)
    for step, loss in {50: 1.441, 100: 1.438, 150: 1.435, 200: 1.426, 250: 1.486, 300: 1.488, 350: 1.493}.items():
        mlflow.log_metric("val_loss", loss, step=step)
    mlflow.log_metrics({"best_val_loss": 1.426, "best_val_step": 200})
    mlflow.set_tag("status", "rank32 - no improvement over rank16")
    mlflow.set_tag("adapters_saved", "yes")
    mlflow.set_tag("adapters_path", "adapters_run5-2ep-lr2e4-r32")
print("Run 5 logged!")

# Verify all runs
runs = mlflow.search_runs(experiment_names=["product-review-summarizer"])
print(f"\nTotal runs in DB: {len(runs)}")
for _, run in runs.iterrows():
    name = run.get('tags.mlflow.runName', 'unnamed')
    best_val = run.get('metrics.best_val_loss', 'N/A')
    status = run.get('tags.status', '')
    print(f"  {name} | best_val: {best_val} | {status}")

MLflow version: 2.19.0


2026/04/24 02:07:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/24 02:07:51 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

Run 1 logged!
Run 2 logged!
Run 3 logged!
Run 4 logged!
Run 4 retrain logged!
Run 5 logged!

Total runs in DB: 6
  run5-2ep-lr2e4-r32 | best_val: 1.426 | rank32 - no improvement over rank16
  run4-2ep-lr1.5e4-r16 | best_val: 1.422 | winner - adapters saved
  run4-2ep-lr1.5e4-r16-prev | best_val: 1.419 | best val loss - adapters lost in crash
  run3-2ep-lr2e4-r16 | best_val: 1.422 | good - adapters lost in crash
  run2-1ep-lr1e4-r16 | best_val: 1.592 | underfit
  run1-3ep-lr2e4-r16 | best_val: 1.475 | overfit


In [2]:
# Install cloudflared again (fresh runtime)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null

import subprocess, time, re, requests

# Start tunnel
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=open("/tmp/cf_out.txt", "w"),
    stderr=open("/tmp/cf_err.txt", "w")
)
time.sleep(15)

with open("/tmp/cf_err.txt", "r") as f:
    output = f.read()
urls = re.findall(r'https://([a-zA-Z0-9-]+\.trycloudflare\.com)', output)

if urls:
    tunnel_domain = urls[0]
    print(f"1. Tunnel ready: {tunnel_domain}")

    # MLflow 2.19 — no --allowed-hosts needed
    mlflow_process = subprocess.Popen(
        ["mlflow", "ui",
         "--host", "0.0.0.0",
         "--port", "5000",
         "--backend-store-uri", "sqlite:////tmp/mlflow_fresh.db"],
        stdout=open("/tmp/mlflow_out.txt", "w"),
        stderr=open("/tmp/mlflow_err.txt", "w")
    )

    for i in range(12):
        time.sleep(5)
        try:
            r = requests.get("http://localhost:5000")
            if r.status_code == 200:
                print(f"2. MLflow ready!")
                print(f"\n{'='*60}")
                print(f"   https://{tunnel_domain}")
                print(f"{'='*60}")
                break
        except:
            print(f"   Attempt {i+1}: Starting...")
else:
    print("Cloudflared failed")
    print(output[:500])

(Reading database ... 118198 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
1. Tunnel ready: merely-bingo-brought-possess.trycloudflare.com
2. MLflow ready!

   https://merely-bingo-brought-possess.trycloudflare.com


Saving this working MLflow DB to Google Drive
Updating the training script to use the same MLflow setup

In [3]:
# Save working MLflow DB to Drive
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/review-summarizer'
!cp /tmp/mlflow_fresh.db {save_dir}/mlflow_fresh.db
print("MLflow DB saved to Drive!")

Mounted at /content/drive
MLflow DB saved to Drive!
